In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip uninstall -y faiss faiss-cpu
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 51.5 MB/s eta 0:00:00


In [3]:
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 3.0 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.5.1
    Uninstalling sentence-transformers-5.5.1:
      Successfully uninstalled sentence-transformers-5.5.1


In [ ]:
!pip install -q transformers accelerate torch python-docx

In [ ]:
import faiss
import pickle
import numpy as np
import pandas as pd

from docx import Document
from sentence_transformers import SentenceTransformer

In [ ]:
# =====================================================
# FILE PATHS
# =====================================================

JD_FILE = '/content/drive/MyDrive/Hackthon/IndiaRuns/India_runs_data_and_ai_challenge/job_description.docx'

FAISS_INDEX_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/India_runs_data_and_ai_challenge/modelrelated/candidatesummaryvectordb/candSummaryvector_db.faiss"
METADATA_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/India_runs_data_and_ai_challenge/modelrelated/candidatesummaryvectordb/candSummarymetadata.pkl"

MODEL_NAME = "all-MiniLM-L6-v2"

# =====================================================
# READ JD DOCX
# =====================================================

def read_docx(file_path):
    doc = Document(file_path)

    text = "\n".join(
        para.text.strip()
        for para in doc.paragraphs
        if para.text.strip()
    )

    return text

jd_text = read_docx(JD_FILE)

print("JD Length:", len(jd_text))
print(jd_text[:500])

# =====================================================
# LOAD MODEL
# =====================================================

model = SentenceTransformer(MODEL_NAME)

# =====================================================
# GENERATE JD EMBEDDING
# =====================================================

jd_embedding = model.encode(
    [jd_text],
    normalize_embeddings=True
).astype(np.float32)

# =====================================================
# LOAD FAISS
# =====================================================

index = faiss.read_index(FAISS_INDEX_FILE)

with open(METADATA_FILE, "rb") as f:
    metadata = pickle.load(f)

print("Candidates:", len(metadata))

# =====================================================
# SEARCH ALL CANDIDATES
# =====================================================

k = index.ntotal

scores, indices = index.search(
    jd_embedding,
    k
)



JD Length: 9564
Job Description: Senior AI Engineer — Founding Team
Company: Redrob AI (Series A AI-native talent intelligence platform)
Location: Pune/Noida, India (Hybrid — flexible cadence) | Open to relocation candidates from Tier-1 Indian cities
Employment Type: Full-time
Experience Required: 5–9 years (see "what we mean by this" below)
Let's be honest about this role
We're going to write this JD differently from most. We're a Series A company that just raised our round and we're building a new AI Engineer


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Candidates: 100000
Saved: /content/drive/MyDrive/Hackthon/IndiaRuns/India_runs_data_and_ai_challenge/modelrelated/candidate_semantic_scores.csv
    candidate_id              name  \
1   CAND_0000388  Tanvi Chatterjee   
0   CAND_0033385         Nisha Sen   
3   CAND_0011223       Dev Trivedi   
2   CAND_0061355       Ishaan Iyer   
4   CAND_0083549      Manish Hegde   
6   CAND_0047811      Sai Krishnan   
5   CAND_0059347        Siya Kumar   
8   CAND_0034391      Suresh Naidu   
9   CAND_0018525      Aanya Shetty   
7   CAND_0093731    Krishna Sharma   
10  CAND_0087167      Aditya Gupta   
20  CAND_0089263      Tanya Saxena   
14  CAND_0048089    Shaurya Pandey   
13  CAND_0058607         Rahul Rao   
18  CAND_0041211      Rajesh Ghosh   
23  CAND_0019965     Myra Krishnan   
12  CAND_0071908     Pranav Pandey   
16  CAND_0068264          Om Verma   
11  CAND_0085263   Myra Chatterjee   
21  CAND_0039597      Deepak Joshi   

                                             headline  ye

In [ ]:
# =====================================================
# BUILD RESULT CSV
# =====================================================

results = []

for rank, idx in enumerate(indices[0]):

    similarity = float(scores[0][rank])

    # cosine similarity -> score 0-1
    semantic_score = (similarity + 1) / 2


    candidate = metadata[idx]

    results.append({
        "candidate_id": candidate.get("candidate_id"),
        "name": candidate.get("name"),
        "headline": candidate.get("headline"),
        "years_of_exp": candidate.get("years_of_exp"),
        "semantic_similarity": round(similarity, 4),
        "semantic_score": semantic_score
    })

# =====================================================
# SAVE OUTPUT
# =====================================================

result_df = pd.DataFrame(results)

result_df = result_df.sort_values(
    "semantic_score",
    ascending=False
)

OUTPUT_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/India_runs_data_and_ai_challenge/modelrelated/candidate_semantic_scores.csv"

result_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(f"Saved: {OUTPUT_FILE}")
print(result_df.head(20))

Saved: /content/drive/MyDrive/Hackthon/IndiaRuns/India_runs_data_and_ai_challenge/modelrelated/candidate_semantic_scores.csv
    candidate_id              name  \
0   CAND_0033385         Nisha Sen   
1   CAND_0000388  Tanvi Chatterjee   
3   CAND_0011223       Dev Trivedi   
2   CAND_0061355       Ishaan Iyer   
5   CAND_0059347        Siya Kumar   
4   CAND_0083549      Manish Hegde   
6   CAND_0047811      Sai Krishnan   
9   CAND_0018525      Aanya Shetty   
7   CAND_0093731    Krishna Sharma   
8   CAND_0034391      Suresh Naidu   
14  CAND_0048089    Shaurya Pandey   
10  CAND_0087167      Aditya Gupta   
13  CAND_0058607         Rahul Rao   
11  CAND_0085263   Myra Chatterjee   
15  CAND_0014461        Arnav Nair   
12  CAND_0071908     Pranav Pandey   
17  CAND_0066536        Sunil Shah   
18  CAND_0041211      Rajesh Ghosh   
19  CAND_0039687    Dev Chatterjee   
16  CAND_0068264          Om Verma   

                                             headline  years_of_exp  \
0    

In [6]:
import json
import pickle
import faiss
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

# =====================================================
# FILE PATHS
# =====================================================

JSON_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/data/cleaned/job_requirements.json"

FAISS_INDEX_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/India_runs_data_and_ai_challenge/modelrelated/candidatesummaryvectordb/candSummaryvector_db.faiss"

METADATA_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/India_runs_data_and_ai_challenge/modelrelated/candidatesummaryvectordb/candSummarymetadata.pkl"

OUTPUT_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/India_runs_data_and_ai_challenge/modelrelated/candidatesummaryvectordb/candidate_semantic_scores.csv"

MODEL_NAME = "all-MiniLM-L6-v2"

# =====================================================
# LOAD JOB REQUIREMENTS JSON
# =====================================================

with open(JSON_FILE, "r") as f:
    jd = json.load(f)

# =====================================================
# BUILD SEMANTIC QUERY
# =====================================================

core_skills = [
    s["skill_id"]
    for s in jd["skills"]["core_required"]
]

secondary_skills = [
    s["skill_id"]
    for s in jd["skills"]["secondary_skills"]
]

domains = [
    d["domain_id"]
    for d in jd["experience_requirements"]["required_domains"]
]

roles = jd["roles"]["preferred_titles"]

industries = jd["company"]["industry_constraints"]["preferred_industries"]

cities = jd["location"]["domestic"]["preferred_cities"]

jd_text = f"""
Hiring for an AI/ML Search Engineer.

Preferred Roles:
{', '.join(roles)}

Mandatory Skills:
{', '.join(core_skills)}

Preferred Skills:
{', '.join(secondary_skills)}

Required Domains:
{', '.join(domains)}

Preferred Industries:
{', '.join(industries)}

Preferred Locations:
{', '.join(cities)}

Experience:
{jd['experience']['min_years']} to {jd['experience']['max_years']} years

Engineering Style:
{jd['work_style']['engineering_style']}

Communication Model:
{jd['work_style']['communication_model']}
"""

print("=" * 100)
print("JD QUERY")
print("=" * 100)
print(jd_text)

# =====================================================
# LOAD EMBEDDING MODEL
# =====================================================

model = SentenceTransformer(MODEL_NAME)

# =====================================================
# GENERATE JD EMBEDDING
# =====================================================

jd_embedding = model.encode(
    [jd_text],
    normalize_embeddings=True
).astype(np.float32)

# =====================================================
# LOAD FAISS INDEX
# =====================================================

index = faiss.read_index(FAISS_INDEX_FILE)

with open(METADATA_FILE, "rb") as f:
    metadata = pickle.load(f)

print(f"Total Candidates: {len(metadata)}")

# =====================================================
# SEARCH ALL CANDIDATES
# =====================================================

k = index.ntotal

scores, indices = index.search(
    jd_embedding,
    k
)

# =====================================================
# BUILD RESULTS
# =====================================================

results = []

for rank, idx in enumerate(indices[0]):

    score = float(scores[0][rank])

    candidate = metadata[idx]

    candidate_id = candidate.get(
        "candidate_id",
        f"CAND_{idx}"
    )

    results.append({
        "candidate_id": candidate_id,
        "semantic_score": round(score, 6)
    })

# =====================================================
# SAVE RESULTS
# =====================================================

result_df = pd.DataFrame(results)

result_df = result_df.sort_values(
    "semantic_score",
    ascending=False
)

result_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\nTop 20 Candidates")
print(result_df.head(20))

print(f"\nSaved to: {OUTPUT_FILE}")

JD QUERY

Hiring for an AI/ML Search Engineer.

Preferred Roles:
SENIOR_AI_ENGINEER, ML_ENGINEER, SEARCH_ENGINEER, RANKING_ENGINEER, RECOMMENDER_SYSTEMS_ENGINEER, APPLIED_SCIENTIST

Mandatory Skills:
EMBEDDINGS, RETRIEVAL_SYSTEMS, VECTOR_DB, HYBRID_SEARCH, PYTHON, RANKING_SYSTEMS_EVALUATION, EVALUATION_FRAMEWORKS, AB_TESTING

Preferred Skills:
LLM_FINE_TUNING, LEARNING_TO_RANK, HR_TECH, DISTRIBUTED_SYSTEMS, OPEN_SOURCE_CONTRIBUTION

Required Domains:
RETRIEVAL_SYSTEMS, RANKING_SYSTEMS, SEARCH_SYSTEMS, ML_PRODUCTION_SYSTEMS

Preferred Industries:
AI_NATIVE_TALENT_INTELLIGENCE, RECRUITMENT_TECH, SEARCH_INFRASTRUCTURE, RECOMMENDATION_SYSTEMS, HR_TECH, APPLIED_ML

Preferred Locations:
PUNE, NOIDA

Experience:
5 to 9 years

Engineering Style:
FULL_STACK_ML_SYSTEMS

Communication Model:
ASYNC_WRITING_HEAVY



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total Candidates: 100000

Top 20 Candidates
    candidate_id  semantic_score
0   CAND_0001930        0.591914
4   CAND_0001302        0.591783
1   CAND_0076412        0.591783
2   CAND_0054271        0.591783
3   CAND_0010655        0.591783
5   CAND_0088602        0.591660
6   CAND_0031143        0.591649
7   CAND_0022463        0.591644
8   CAND_0066035        0.591464
10  CAND_0041083        0.591377
9   CAND_0069638        0.591377
11  CAND_0035315        0.591310
12  CAND_0025109        0.591118
13  CAND_0078785        0.591079
14  CAND_0003791        0.591004
15  CAND_0053934        0.590956
18  CAND_0010408        0.590956
17  CAND_0009200        0.590956
16  CAND_0028287        0.590956
20  CAND_0049163        0.590942

Saved to: /content/drive/MyDrive/Hackthon/IndiaRuns/India_runs_data_and_ai_challenge/modelrelated/candidatesummaryvectordb/candidate_semantic_scores.csv
